In [5]:
import illustris_python as il
import requests
import h5py
import io
import numpy as np

In [90]:
API_KEY = "624abff74a436f05f87b576b68b19a76"
BASE_URL = "https://www.tng-project.org/api"
SIMULATION = "TNG50-1"
SNAPSHOT = 99
TARGET_SUBFIND_ID = 441709 #ids = np.array([441709, 474008, 532301])

HEADERS = {"api-key": API_KEY}

# --- Test 1: Check the root API endpoint first ---
r = requests.get(f"{BASE_URL}/", headers=HEADERS)
print("Status:", r.status_code)
print("Content-Type:", r.headers.get('Content-Type'))

if 'application/json' in r.headers.get('Content-Type', ''):
    print("Root API is working correctly!")
else:
    print("Something is wrong — expected JSON, got:", r.content[:300])

Status: 200
Content-Type: application/json
Root API is working correctly!


In [91]:
r2 = requests.get(
    f"{BASE_URL}/{SIMULATION}/snapshots/{SNAPSHOT}/subhalos/{TARGET_SUBFIND_ID}/",
    headers=HEADERS
)
print(r2.status_code)
print(r2.headers.get('Content-Type'))
print(r2.text[:500])

200
application/json
{"snap":99,"id":441709,"bhmdot":1.71616e-05,"cm_x":7688.08,"cm_y":7903.81,"cm_z":12123.0,"gasmetallicity":0.0226147,"gasmetallicityhalfrad":0.0292039,"gasmetallicitymaxrad":0.0,"gasmetallicitysfr":0.0221671,"gasmetallicitysfrweighted":0.0234962,"pos_x":7702.16,"pos_y":7901.68,"pos_z":12109.9,"halfmassrad":102.324,"halfmassrad_gas":132.921,"halfmassrad_dm":106.02,"halfmassrad_stars":4.9427,"halfmassrad_bhs":0.0,"len":10397204,"len_gas":2526044,"len_dm":5530765,"len_stars":2340394,"len_bhs":1,"mas


In [92]:
import json
print(json.dumps(r2.json(), indent=2))
tree_url = r2.json()['trees']['sublink']

{
  "snap": 99,
  "id": 441709,
  "bhmdot": 1.71616e-05,
  "cm_x": 7688.08,
  "cm_y": 7903.81,
  "cm_z": 12123.0,
  "gasmetallicity": 0.0226147,
  "gasmetallicityhalfrad": 0.0292039,
  "gasmetallicitymaxrad": 0.0,
  "gasmetallicitysfr": 0.0221671,
  "gasmetallicitysfrweighted": 0.0234962,
  "pos_x": 7702.16,
  "pos_y": 7901.68,
  "pos_z": 12109.9,
  "halfmassrad": 102.324,
  "halfmassrad_gas": 132.921,
  "halfmassrad_dm": 106.02,
  "halfmassrad_stars": 4.9427,
  "halfmassrad_bhs": 0.0,
  "len": 10397204,
  "len_gas": 2526044,
  "len_dm": 5530765,
  "len_stars": 2340394,
  "len_bhs": 1,
  "mass": 195.077,
  "mass_gas": 15.5699,
  "mass_dm": 169.998,
  "mass_stars": 9.4771,
  "mass_bhs": 0.032407,
  "massinhalfrad": 8.82923,
  "massinhalfrad_gas": 0.397986,
  "massinhalfrad_dm": 3.66029,
  "massinhalfrad_stars": 4.73855,
  "massinhalfrad_bhs": 0.032407,
  "massinmaxrad": 4.5e-05,
  "massinmaxrad_gas": 0.0,
  "massinmaxrad_dm": 0.0,
  "massinmaxrad_stars": 4.5e-05,
  "massinmaxrad_bhs": 0

In [93]:
r3 = requests.get(tree_url, headers=HEADERS)
r3.raise_for_status()
# print("Status:", r3.status_code)
# print("Content-Type:", r3.headers.get('Content-Type'))
# print("Content-Length:", r3.headers.get('Content-Length'))
# print("First 16 bytes:", r3.content[:16])  # Should start with b'\x89HDF'

In [94]:
# Load HDF5 from memory
with h5py.File(io.BytesIO(r3.content), 'r') as f:
    print("Keys in tree file:", list(f.keys()))  # Let's see what fields are available
    subhalo_ids    = f['SubhaloID'][:]
    subfind_ids    = f['SubfindID'][:]
    snap_nums      = f['SnapNum'][:]
    first_prog_ids = f['FirstProgenitorID'][:]
    next_prog_ids  = f['NextProgenitorID'][:]
    masses         = f['SubhaloMassType'][:]  # shape (N, 6), index 4 = stellar

Keys in tree file: ['DescendantID', 'FirstProgenitorID', 'FirstSubhaloInFOFGroupID', 'GroupBHMass', 'GroupBHMdot', 'GroupCM', 'GroupFirstSub', 'GroupGasMetalFractions', 'GroupGasMetallicity', 'GroupLen', 'GroupLenType', 'GroupMass', 'GroupMassType', 'GroupNsubs', 'GroupPos', 'GroupSFR', 'GroupStarMetalFractions', 'GroupStarMetallicity', 'GroupVel', 'GroupWindMass', 'Group_M_Crit200', 'Group_M_Crit500', 'Group_M_Mean200', 'Group_M_TopHat200', 'Group_R_Crit200', 'Group_R_Crit500', 'Group_R_Mean200', 'Group_R_TopHat200', 'LastProgenitorID', 'MainLeafProgenitorID', 'Mass', 'MassHistory', 'NextProgenitorID', 'NextSubhaloInFOFGroupID', 'NumParticles', 'RootDescendantID', 'SnapNum', 'SubfindID', 'SubhaloBHMass', 'SubhaloBHMdot', 'SubhaloCM', 'SubhaloGasMetalFractions', 'SubhaloGasMetalFractionsHalfRad', 'SubhaloGasMetalFractionsMaxRad', 'SubhaloGasMetalFractionsSfr', 'SubhaloGasMetalFractionsSfrWeighted', 'SubhaloGasMetallicity', 'SubhaloGasMetallicityHalfRad', 'SubhaloGasMetallicityMaxRad', 

In [95]:
print(f"Total nodes in tree: {len(subhalo_ids)}")
print(f"Root SubfindID: {subfind_ids[0]} at snap {snap_nums[0]}")  # Should be 532301 at snap 99


Total nodes in tree: 165821
Root SubfindID: 441709 at snap 99


In [96]:
# Build lookup table: SubhaloID → array index
id_to_idx = {sid: i for i, sid in enumerate(subhalo_ids)}

In [97]:
# Walk the tree recursively to find all merger progenitors
def walk_tree(idx):
    mergers = []

    fp_id = first_prog_ids[idx]
    if fp_id == -1 or fp_id not in id_to_idx:
        return mergers  # Leaf node, no progenitors

    fp_idx = id_to_idx[fp_id]

    # Recurse down the main branch first
    mergers += walk_tree(fp_idx)

    # Collect all secondary progenitors (mergers) via NextProgenitorID
    np_id = next_prog_ids[fp_idx]
    while np_id != -1 and np_id in id_to_idx:
        np_idx = id_to_idx[np_id]
        mergers.append({
            'SubfindID':   int(subfind_ids[np_idx]),
            'SnapNum':     int(snap_nums[np_idx]),
            'StellarMass': float(masses[np_idx, 4])  # in 1e10 M_sun/h
        })
        mergers += walk_tree(np_idx)
        np_id = next_prog_ids[np_idx]

    return mergers

In [98]:
import sys
sys.setrecursionlimit(10000)  # Needed for deep trees

mergers = walk_tree(0)

target_stellar_mass = float(masses[0, 4])  # root subhalo stellar mass

real_mergers = [m for m in mergers if m['StellarMass'] > 0.01]

print(f"Total merger progenitors:          {len(mergers)}")
print(f"With non-trivial stellar mass:     {len(real_mergers)}")
print(f"Target galaxy M_* = {target_stellar_mass:.4f} x 1e10 M_sun/h\n")

for m in sorted(real_mergers, key=lambda x: x['SnapNum']):
    ratio = m['StellarMass'] / target_stellar_mass if target_stellar_mass > 0 else 0
    print(f"  Snap {m['SnapNum']:>3d} | SubfindID {m['SubfindID']:>8d} | "
          f"M_* = {m['StellarMass']:.4f} | mass ratio = 1:{1/ratio:.1f}")

# print(f"Found {len(mergers)} merger progenitors\n")
# for m in sorted(mergers, key=lambda x: x['SnapNum']):
#     print(f"  Snap {m['SnapNum']:>3d} | SubfindID {m['SubfindID']:>8d} | M_* = {m['StellarMass']:.4f} x 1e10 M_sun/h")


Total merger progenitors:          11825
With non-trivial stellar mass:     26
Target galaxy M_* = 9.4771 x 1e10 M_sun/h

  Snap  27 | SubfindID   218840 | M_* = 0.0154 | mass ratio = 1:615.9
  Snap  31 | SubfindID   238713 | M_* = 0.0237 | mass ratio = 1:400.0
  Snap  44 | SubfindID   438246 | M_* = 0.0344 | mass ratio = 1:275.3
  Snap  56 | SubfindID   338493 | M_* = 0.0399 | mass ratio = 1:237.6
  Snap  65 | SubfindID   369512 | M_* = 0.0283 | mass ratio = 1:334.5
  Snap  66 | SubfindID   421725 | M_* = 0.0104 | mass ratio = 1:909.4
  Snap  73 | SubfindID   347096 | M_* = 1.8691 | mass ratio = 1:5.1
  Snap  83 | SubfindID   366384 | M_* = 0.0108 | mass ratio = 1:878.2
  Snap  84 | SubfindID   376169 | M_* = 0.0146 | mass ratio = 1:650.7
  Snap  84 | SubfindID   376176 | M_* = 0.0115 | mass ratio = 1:825.9
  Snap  84 | SubfindID   376170 | M_* = 0.0149 | mass ratio = 1:637.4
  Snap  85 | SubfindID   382486 | M_* = 0.0364 | mass ratio = 1:260.0
  Snap  86 | SubfindID   383335 | M_* = 

In [99]:
root_pos = (r2.json()['pos_x'], r2.json()['pos_y'], r2.json()['pos_z'])

for m in sorted(real_mergers, key=lambda x: x['SnapNum']):
    r = requests.get(
        f"{BASE_URL}/{SIMULATION}/snapshots/{m['SnapNum']}/subhalos/{m['SubfindID']}/",
        headers=HEADERS
    )
    data = r.json()
    
    # Note: comparing positions across different snapshots isn't physically meaningful
    # for early mergers, but is useful for the late ones (snaps 78/79) near z~0.2
    dx = data['pos_x'] - root_pos[0]
    dy = data['pos_y'] - root_pos[1]
    dz = data['pos_z'] - root_pos[2]
    sep = np.sqrt(dx**2 + dy**2 + dz**2)
    
    print(f"SubfindID {m['SubfindID']} at Snap {m['SnapNum']}:")
    print(f"  Position: ({data['pos_x']:.2f}, {data['pos_y']:.2f}, {data['pos_z']:.2f}) ckpc/h")
    print(f"  Separation from root at snap 99: {sep:.2f} ckpc/h")
    print()

SubfindID 218840 at Snap 27:
  Position: (8703.48, 7692.47, 12166.60) ckpc/h
  Separation from root at snap 99: 1024.51 ckpc/h

SubfindID 238713 at Snap 31:
  Position: (8632.17, 7641.72, 12208.40) ckpc/h
  Separation from root at snap 99: 970.67 ckpc/h

SubfindID 438246 at Snap 44:
  Position: (8819.47, 7490.51, 12709.10) ckpc/h
  Separation from root at snap 99: 1332.85 ckpc/h

SubfindID 338493 at Snap 56:
  Position: (8187.41, 7453.11, 12532.60) ckpc/h
  Separation from root at snap 99: 784.45 ckpc/h

SubfindID 369512 at Snap 65:
  Position: (8089.97, 7558.17, 12341.40) ckpc/h
  Separation from root at snap 99: 567.44 ckpc/h

SubfindID 421725 at Snap 66:
  Position: (7685.34, 8019.99, 12160.00) ckpc/h
  Separation from root at snap 99: 129.58 ckpc/h

SubfindID 347096 at Snap 73:
  Position: (7732.03, 7964.77, 12128.60) ckpc/h
  Separation from root at snap 99: 72.27 ckpc/h

SubfindID 366384 at Snap 83:
  Position: (7771.11, 7841.95, 12171.20) ckpc/h
  Separation from root at snap 99

In [100]:
snap_to_root_subfind = {}
idx = 0  # start at root
while True:
    snap_to_root_subfind[int(snap_nums[idx])] = int(subfind_ids[idx])
    fp_id = first_prog_ids[idx]
    if fp_id == -1 or fp_id not in id_to_idx:
        break
    idx = id_to_idx[fp_id]

print(f"Main progenitor branch has {len(snap_to_root_subfind)} snapshots")

r_snaps = requests.get(f"{BASE_URL}/{SIMULATION}/snapshots/", headers=HEADERS)
snaps_data = r_snaps.json()

# Build a lookup: SnapNum → redshift
snap_to_z = {s['number']: s['redshift'] for s in snaps_data}

# # Test it
# for snap in sorted(snap_to_root_subfind.keys()):
#     print(f"Snap {snap}: z = {snap_to_z[snap]:.4f}")

Main progenitor branch has 99 snapshots


In [101]:
h = 0.6774  # TNG Hubble parameter

for m in sorted(real_mergers, key=lambda x: x['SnapNum']):
    snap = m['SnapNum']
    z = snap_to_z[snap]
    a = 1 / (1 + z)  # scale factor

    # Get merger galaxy position
    r_merger = requests.get(
        f"{BASE_URL}/{SIMULATION}/snapshots/{snap}/subhalos/{m['SubfindID']}/",
        headers=HEADERS
    )
    merger_data = r_merger.json()

    # Get root galaxy position at the same snapshot
    root_sfid_at_snap = snap_to_root_subfind.get(snap)
    r_root = requests.get(
        f"{BASE_URL}/{SIMULATION}/snapshots/{snap}/subhalos/{root_sfid_at_snap}/",
        headers=HEADERS
    )
    root_data = r_root.json()

    # Separation in comoving kpc/h
    dx = merger_data['pos_x'] - root_data['pos_x']
    dy = merger_data['pos_y'] - root_data['pos_y']
    dz = merger_data['pos_z'] - root_data['pos_z']
    sep_ckpch = np.sqrt(dx**2 + dy**2 + dz**2)

    # Convert to physical kpc: multiply by a, divide by h
    sep_pkpc = sep_ckpch * a #/ h

    print(f"SubfindID {m['SubfindID']} at Snap {snap} (z = {z:.4f}):")
    print(f"  Merger pos:    ({a*merger_data['pos_x']:.2f}, {a*merger_data['pos_y']:.2f}, {a*merger_data['pos_z']:.2f}) kpc/h")
    print(f"  Root pos:      ({a*root_data['pos_x']:.2f}, {a*root_data['pos_y']:.2f}, {a*root_data['pos_z']:.2f}) kpc/h")
    print(f"  Separation:    {sep_ckpch:.2f} ckpc/h  →  {sep_pkpc:.2f} physical kpc/h")
    print()

SubfindID 218840 at Snap 27 (z = 2.7331):
  Merger pos:    (2331.41, 2060.59, 3259.08) kpc/h
  Root pos:      (2237.89, 1934.62, 3512.88) kpc/h
  Separation:    1113.89 ckpc/h  →  298.38 physical kpc/h

SubfindID 238713 at Snap 31 (z = 2.2079):
  Merger pos:    (2690.89, 2382.14, 3805.70) kpc/h
  Root pos:      (2601.05, 2252.25, 4060.44) kpc/h
  Separation:    961.50 ckpc/h  →  299.73 physical kpc/h

SubfindID 438246 at Snap 44 (z = 1.2485):
  Merger pos:    (3922.43, 3331.38, 5652.33) kpc/h
  Root pos:      (3684.89, 3253.25, 5675.59) kpc/h
  Separation:    564.66 ckpc/h  →  251.13 physical kpc/h

SubfindID 338493 at Snap 56 (z = 0.7911):
  Merger pos:    (4571.24, 4161.27, 6997.28) kpc/h
  Root pos:      (4569.91, 4160.61, 6996.77) kpc/h
  Separation:    2.81 ckpc/h  →  1.57 physical kpc/h

SubfindID 369512 at Snap 65 (z = 0.5464):
  Merger pos:    (5231.51, 4887.62, 7980.77) kpc/h
  Root pos:      (5235.60, 4887.94, 7984.91) kpc/h
  Separation:    9.01 ckpc/h  →  5.83 physical kpc/

In [88]:
from astropy.cosmology import FlatLambdaCDM

# TNG cosmological parameters
cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089)

In [89]:
def snap_to_lookback(snap):
    """Returns the lookback time in Gyr for a given snapshot number."""
    z = snap_to_z[snap]
    return cosmo.lookback_time(z).value  # in Gyr

def snap_to_age(snap):
    """Returns the age of the universe in Gyr at a given snapshot number."""
    z = snap_to_z[snap]
    return cosmo.age(z).value  # in Gyr

In [70]:
def get_progenitor_history(snap, subfind_id):
    """
    Given a snap number and subfind ID, returns a list of dicts containing
    the SubfindID and SnapNum for all main progenitors going back in time.
    """
    # Fetch the subhalo metadata to get the tree URL
    r_meta = requests.get(
        f"{BASE_URL}/{SIMULATION}/snapshots/{snap}/subhalos/{subfind_id}/",
        headers=HEADERS
    )
    r_meta.raise_for_status()
    tree_url = r_meta.json()['trees']['sublink_mpb']  # mpb = main progenitor branch only

    # Fetch the tree
    r_tree = requests.get(tree_url, headers=HEADERS)
    r_tree.raise_for_status()

    # Load and walk
    history = []
    with h5py.File(io.BytesIO(r_tree.content), 'r') as f:
        sfids = f['SubfindID'][:]
        snaps = f['SnapNum'][:]
        for sfid, sn in zip(sfids, snaps):
            history.append({
                'SubfindID': int(sfid),
                'SnapNum':   int(sn),
                'Redshift':  snap_to_z[int(sn)],
                'Age_Gyr':   snap_to_age(int(sn))
            })

    return history  # Already ordered from present → past

# Test it on your merger at snap 40
history = get_progenitor_history(40, 320118)
for h in history:
    print(f"  Snap {h['SnapNum']:>3d} | z = {h['Redshift']:.4f} | "
          f"Age = {h['Age_Gyr']:.2f} Gyr | SubfindID {h['SubfindID']}")

  Snap  40 | z = 1.4955 | Age = 4.29 Gyr | SubfindID 320118
  Snap  39 | z = 1.5312 | Age = 4.21 Gyr | SubfindID 318065
  Snap  38 | z = 1.6042 | Age = 4.04 Gyr | SubfindID 311465
  Snap  37 | z = 1.6667 | Age = 3.90 Gyr | SubfindID 302848
  Snap  36 | z = 1.7436 | Age = 3.74 Gyr | SubfindID 295199
  Snap  35 | z = 1.8227 | Age = 3.59 Gyr | SubfindID 290524
  Snap  34 | z = 1.9041 | Age = 3.45 Gyr | SubfindID 298126
  Snap  33 | z = 2.0020 | Age = 3.28 Gyr | SubfindID 290159
  Snap  32 | z = 2.1033 | Age = 3.13 Gyr | SubfindID 285724
  Snap  31 | z = 2.2079 | Age = 2.98 Gyr | SubfindID 371609
  Snap  30 | z = 2.3161 | Age = 2.84 Gyr | SubfindID 358175
  Snap  29 | z = 2.4442 | Age = 2.68 Gyr | SubfindID 344976
  Snap  28 | z = 2.5773 | Age = 2.54 Gyr | SubfindID 330806
  Snap  27 | z = 2.7331 | Age = 2.38 Gyr | SubfindID 314909
  Snap  26 | z = 2.8958 | Age = 2.24 Gyr | SubfindID 294812
  Snap  25 | z = 3.0081 | Age = 2.15 Gyr | SubfindID 281596
  Snap  24 | z = 3.2830 | Age = 1.94 Gyr

In [74]:
get_progenitor_history(99,532301)

[{'SubfindID': 532301,
  'SnapNum': 99,
  'Redshift': 2.22044604925031e-16,
  'Age_Gyr': np.float64(13.802718326812728)},
 {'SubfindID': 530491,
  'SnapNum': 98,
  'Redshift': 0.00952166696794476,
  'Age_Gyr': np.float64(13.666229994235184)},
 {'SubfindID': 524534,
  'SnapNum': 97,
  'Redshift': 0.0239744283827625,
  'Age_Gyr': np.float64(13.462643721151078)},
 {'SubfindID': 522802,
  'SnapNum': 96,
  'Redshift': 0.0337243718735154,
  'Age_Gyr': np.float64(13.32769699692255)},
 {'SubfindID': 517646,
  'SnapNum': 95,
  'Redshift': 0.0485236299818059,
  'Age_Gyr': np.float64(13.126465809714055)},
 {'SubfindID': 513866,
  'SnapNum': 94,
  'Redshift': 0.058507322794513,
  'Age_Gyr': np.float64(12.993117459465955)},
 {'SubfindID': 507039,
  'SnapNum': 93,
  'Redshift': 0.0736613846564387,
  'Age_Gyr': np.float64(12.79432602644711)},
 {'SubfindID': 506070,
  'SnapNum': 92,
  'Redshift': 0.0838844307974793,
  'Age_Gyr': np.float64(12.662632110653012)},
 {'SubfindID': 499856,
  'SnapNum': 91,


In [104]:
print(snap_to_lookback(97),
snap_to_lookback(98),
snap_to_lookback(79))

0.34007460566165193 0.1364883325775459 3.2684494629903362
